In [1]:
import praw

In [2]:
import os
os.chdir( os.path.dirname( os.getcwd() ) )

In [3]:
reddit = praw.Reddit('cyclistNerd') # using auth info from praw.ini

Version 7.5.0 of praw is outdated. Version 7.8.1 was released Friday October 25, 2024.


In [4]:
from collections import defaultdict

In [6]:
subreddit_name = 'photocritique'
post_limit     = 1000

In [9]:
user_activity = defaultdict(lambda: {
    'posts': 0,
    'comments': 0,
    'timestamps': []
})

print(f"Analyzing r/{subreddit_name}...")
print(f"Fetching {post_limit} recent posts and their comments...\n")

posts_processed = 0
comments_processed = 0

# Iterate through recent posts
for submission in reddit.subreddit(subreddit_name).new(limit=post_limit):
    posts_processed += 1

    # Track the post author
    if submission.author is not None:
        username = submission.author.name
        user_activity[username]['posts'] += 1
        user_activity[username]['timestamps'].append(submission.created_utc)

    # Get all comments for this post
    # replace_more() fetches all nested comments
    submission.comments.replace_more(limit=0)  # limit=0 gets all comments

    # Flatten the comment tree and iterate through all comments
    for comment in submission.comments.list():
        comments_processed += 1
        if comment.author is not None:
            username = comment.author.name
            user_activity[username]['comments'] += 1
            user_activity[username]['timestamps'].append(comment.created_utc)

    if posts_processed % 10 == 0:
        print(f"Processed {posts_processed}/{post_limit} posts...")

print(f"\nFinished processing {posts_processed} posts with {comments_processed:,d} comments.")

Analyzing r/photocritique...
Fetching 1000 recent posts and their comments...

Processed 10/1000 posts...
Processed 20/1000 posts...
Processed 30/1000 posts...
Processed 40/1000 posts...
Processed 50/1000 posts...
Processed 60/1000 posts...
Processed 70/1000 posts...
Processed 80/1000 posts...
Processed 90/1000 posts...
Processed 100/1000 posts...
Processed 110/1000 posts...
Processed 120/1000 posts...
Processed 130/1000 posts...
Processed 140/1000 posts...
Processed 150/1000 posts...
Processed 160/1000 posts...
Processed 170/1000 posts...
Processed 180/1000 posts...
Processed 190/1000 posts...
Processed 200/1000 posts...
Processed 210/1000 posts...
Processed 220/1000 posts...
Processed 230/1000 posts...
Processed 240/1000 posts...
Processed 250/1000 posts...
Processed 260/1000 posts...
Processed 270/1000 posts...
Processed 280/1000 posts...
Processed 290/1000 posts...
Processed 300/1000 posts...
Processed 310/1000 posts...
Processed 320/1000 posts...
Processed 330/1000 posts...
Proces

# store

In [12]:
import pandas as pd

In [10]:
len(user_activity)

3259

In [11]:
user_activity['cyclistNerd']

{'posts': 1,
 'comments': 9,
 'timestamps': [1762396649.0,
  1762396747.0,
  1762392684.0,
  1762377213.0,
  1762296132.0,
  1762282818.0,
  1762284574.0,
  1762284756.0,
  1762285131.0,
  1762280382.0]}

In [14]:
all_timestamps = []
for user in user_activity:
    for timestamp in user_activity[user]['timestamps']:
        all_timestamps.append(timestamp)

In [15]:
len(all_timestamps)

14252

In [16]:
pd.Timestamp(min(all_timestamps), unit='s'), pd.Timestamp(max(all_timestamps), unit='s')

(Timestamp('2025-10-04 18:41:55'), Timestamp('2025-11-09 21:22:51'))

In [17]:
import json

In [18]:
print(
    json.dumps(user_activity['cyclistNerd'])
)

{"posts": 1, "comments": 9, "timestamps": [1762396649.0, 1762396747.0, 1762392684.0, 1762377213.0, 1762296132.0, 1762282818.0, 1762284574.0, 1762284756.0, 1762285131.0, 1762280382.0]}


In [20]:
with open('extension_helper/recent_user_activity.jsonl', 'w') as f:
    for user, activity in user_activity.items():
        f.write(
            json.dumps(
                {
                    'username'   : user,
                    'posts'      : activity['posts'],
                    'comments'   : activity['comments'],
                    'timestamps' : [int(t) for t in activity['timestamps']],
                }
            ) + '\n'
        )